# Unified Tier-Sweep Experiment

Single self-contained cell: runs the full ROI pipeline across model tiers x seeds and writes `experiments/results/tier_comparison.json`. Edit the `CONFIG` dict, then Run All.


In [1]:
# =============================================================================
# UNIFIED TIER-SWEEP EXPERIMENT  --  single-cell, self-contained
# Automating Interview-Based Generative-AI ROI Measurement (DSRM artifact)
#
# Runs the FULL pipeline (Agent 1 -> process graph -> Agent 2 -> Agent 3 ->
# five-axis validation) across model tiers, repeated over seeds, and writes
# experiments/results/tier_comparison.{json,csv}. Everything is in THIS one cell.
#
# Before running:
#   * .env at the repo root: OPENAI_API_KEY + LLM_MODEL_WEAK/MID/STRONG
#   * launch Jupyter from the repo root
# Config is the CONFIG dict just below.
# =============================================================================

import statistics
import itertools
import collections
from dataclasses import dataclass, field, asdict

# =============================================================================
# NOTEBOOK: 01_agent1_task_extraction.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# AGENT 1 of 3 — Task Extraction & Automatability Classification
#   Input : raw operational interview transcript (any domain)
#   Method: Chain-of-Thought (Wei et al., 2022) — reason step-by-step, then emit
#           a structured list of work units, each with an AI-automatability grade
#           (full / partial / manual) AND an explicit rationale.
#   Output: artifacts/inference/agent1_work_units.json  (list[WorkUnit])
#
# Design principles realized here:
#   DP1 (role separation): this agent ONLY extracts + classifies. It never
#        estimates time (Agent 2) or computes ROI (Agent 3).
#   DP3 (mark the unknown): any field not stated in the interview is set to the
#        MISSING sentinel rather than guessed.
#   Transparency axis: every automatability grade carries a rationale string, so
#        a human reviewer can audit *why* — not just *what*.
#
# All example content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Re-establish foundation from Notebook 00 (self-contained bootstrap)
#
# Notebooks cannot share live Python objects, so we rebuild the minimal set of
# paths, API client, schema, and llm_call() here. This mirrors 00 exactly and
# lets Notebook 01 run standalone.
# =============================================================================
import os
import re
import json
import time
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# --- Paths -------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW  = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
TAB       = ARTIFACTS / "tables"
CACHE     = ARTIFACTS / "llm_cache"
for p in (INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


SEED = 42
np.random.seed(SEED)

# --- Load run manifest written by Notebook 00 --------------------------------
RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
if not RUN_MANIFEST.exists():
    raise FileNotFoundError(
        "[ERROR] artifacts/run_manifest.json not found. "
        "Run 00_setup_and_data.ipynb first."
    )
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))

MODELS       = manifest["models"]
DEFAULT_TIER = manifest["default_tier"]
AUTO_GRADES  = manifest["auto_grades"]
MISSING      = manifest["missing_sentinel"]

print(f"[INFO] Manifest loaded. default_tier={DEFAULT_TIER}, "
      f"grades={list(AUTO_GRADES)}")

# --- API client --------------------------------------------------------------
load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env at project root.")
client = OpenAI(api_key=OPENAI_API_KEY)


# %%
# =============================================================================
# Cell 2. Re-declare schema, cost tracker, and llm_call() (identical to nb 00)
# =============================================================================
@dataclass
class WorkUnit:
    """One extracted task. Domain-neutral fields only."""
    id: str
    name: str
    actor: str = MISSING
    system: str = MISSING
    description: str = ""
    auto_grade: str = MISSING
    auto_rationale: str = ""
    minutes_per_case: Any = MISSING
    cases_per_month: Any = MISSING
    monthly_minutes: Any = MISSING
    time_rationale: str = ""
    evidence: str = ""

    def to_dict(self) -> dict:
        return asdict(self)


class CostTracker:
    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier, model, usage, tag=""):
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        cached = 0
        det = getattr(usage, "prompt_tokens_details", None)
        if det is not None:
            cached = getattr(det, "cached_tokens", 0) or 0
        fresh = max(pt - cached, 0)
        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + ct * pout) / 1_000_000
        self.records.append({
            "tag": tag, "tier": tier, "model": model,
            "prompt_tokens": pt, "cached_tokens": cached,
            "completion_tokens": ct, "cost_usd": cost,
        })
        return cost or 0.0

    def summary(self):
        return pd.DataFrame(self.records) if self.records else pd.DataFrame()

    def total_usd(self):
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))

    def flush(self, stage: str):
        """Append this notebook's spend to a shared ledger so the five-axis
        Efficiency computation and the dashboard can read the TRUE cumulative
        pipeline cost across all notebooks (they do not share memory)."""
        ledger = ARTIFACTS / "cost_ledger.json"
        data = json.loads(ledger.read_text(encoding="utf-8")) if ledger.exists() else {}
        data[stage] = {
            "total_usd": round(self.total_usd(), 6),
            "n_calls": len(self.records),
        }
        ledger.write_text(json.dumps(data, ensure_ascii=False, indent=2),
                          encoding="utf-8")
        return data


COST = CostTracker()


def _safe_json(text):
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json, salt=""):
    raw = json.dumps({"m": model, "s": system, "u": user,
                      "t": temperature, "j": response_json, "salt": salt},
                     ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0,
             response_json=True, tag="", use_cache=True, max_retries=3, salt=""):
    model = MODELS[tier]["name"]
    # cache/seed controls for independent repeats
    if _RUN_DISABLE_CACHE:
        use_cache = False
    try:
        salt = f"{salt}|seed={_RUN_SEED}"
    except NameError:
        pass
    key = _cache_key(model, system, user, temperature, response_json, salt)
    cache_file = CACHE / f"{key}.json"
    if use_cache and cache_file.exists():
        c = json.loads(cache_file.read_text(encoding="utf-8"))
        c["cached"] = True
        c["cost_usd"] = 0.0
        return c
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = _safe_json(text) if response_json else None
            if response_json and parsed is None:
                raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {"text": text, "json": parsed, "cached": False,
                   "tier": tier, "model": model, "cost_usd": cost}
            if use_cache:
                cache_file.write_text(json.dumps(out, ensure_ascii=False),
                                      encoding="utf-8")
            return out
        except TypeError as e:
            if "temperature" in kwargs:
                kwargs.pop("temperature", None)
                last_err = e
                continue
            last_err = e
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] Schema, CostTracker, llm_call() re-established for nb 01.")

# manifest-derived globals used by merged agent logics (nb03/nb02)
COST_MODEL = manifest["cost_model"]
N_ROLLOUTS = manifest["pipeline_cfg"]["agent2_time"]["n_samples"]
SC_TEMPERATURE = manifest["pipeline_cfg"]["agent2_time"]["temperature"]



# ----------------------------- CONFIG ----------------------------------------
CONFIG = {
    "tiers": ["weak", "mid", "strong"],   # model tiers to sweep
    "seeds": [42, 43, 44, 45, 46],        # repeats per tier (independent)
    "matcher_tier": "strong",             # accuracy-axis matcher held fixed
    "disable_cache": True,                # REQUIRED for independent repeats
    "smoke_test": False,                  # True -> only tiers[:1], seeds[:2]
}
# -----------------------------------------------------------------------------


import os as _os, json as _json, shutil as _shutil, statistics as _stats, csv as _csv
from pathlib import Path as _Path
from datetime import datetime as _dt, timezone as _tz

_ROOT = _Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
    _os.chdir(_ROOT)
_INFER   = _ROOT / "artifacts" / "inference"
_MANIF   = _ROOT / "artifacts" / "run_manifest.json"
_RESULTS = _ROOT / "experiments" / "results"
_RESULTS.mkdir(parents=True, exist_ok=True)
assert (_ROOT/".env").exists(), f".env not found at {_ROOT}"
_ORIG_MANIFEST = _MANIF.read_text(encoding="utf-8")

_tiers = CONFIG["tiers"][:1] if CONFIG["smoke_test"] else CONFIG["tiers"]
_seeds = CONFIG["seeds"][:2] if CONFIG["smoke_test"] else CONFIG["seeds"]
_MATCHER_TIER = CONFIG["matcher_tier"]

# The five agent-logic blocks are stored as strings and exec'd in this module's
# globals so they see the bootstrap (llm_call, INFER, manifest, ...). Between
# agents they communicate via artifacts/inference/*.json exactly as before.
_AGENT_LOGIC = {
    "01_agent1_task_extraction": "# %%\n# =============================================================================\n# Cell 3. Load interview text (the sole domain input to Agent 1)\n# =============================================================================\nINTERVIEW_PATH = DATA_RAW / manifest[\"paths\"][\"interview\"].split(\"/\")[-1] \\\n    if \"/\" in manifest[\"paths\"][\"interview\"] else DATA_RAW / \"interview_tcb.txt\"\n# Robust fallback to the known filename:\nif not INTERVIEW_PATH.exists():\n    INTERVIEW_PATH = DATA_RAW / \"interview_tcb.txt\"\n\nINTERVIEW_TEXT = INTERVIEW_PATH.read_text(encoding=\"utf-8\").strip()\nprint(f\"[INFO] Interview: {len(INTERVIEW_TEXT)} chars from {rel(INTERVIEW_PATH)}\")\n\n\n# %%\n# =============================================================================\n# Cell 4. Agent-1 prompt (domain-agnostic; Chain-of-Thought + strict JSON)\n#\n# The prompt describes a GENERIC task \u2014 \"extract repeatable work units from any\n# operational interview and grade their AI-automatability\" \u2014 with no reference\n# to the specific domain. Swapping the interview swaps the domain; the prompt is\n# unchanged. This is what makes the artifact transferable.\n# =============================================================================\nAGENT1_SYSTEM = f\"\"\"\\\nYou are Agent 1 (Task Extraction & Automatability Classification) in a pipeline\nthat estimates the ROI of adopting generative AI for knowledge work. You analyze\nan operational interview transcript from ANY business domain and identify the\ndiscrete, repeatable work units it describes.\n\nReason step by step (chain-of-thought) BEFORE producing the final answer:\n  1. Read the transcript and list every distinct task an actor performs.\n  2. Merge duplicates; split compound tasks into atomic work units.\n  3. For each unit, decide who performs it and what system/tool is used, IF and\n     ONLY IF the transcript states it. If not stated, use the sentinel \"{MISSING}\".\n  4. Grade each unit's AI-automatability using exactly one of:\n       - \"full\"    : {AUTO_GRADES['full']}\n       - \"partial\" : {AUTO_GRADES['partial']}\n       - \"manual\"  : {AUTO_GRADES['manual']}\n  5. Write a one-sentence rationale justifying the grade, grounded in the task's\n     nature (data-driven vs. judgment vs. physical), not in domain assumptions.\n\nRules:\n  - Do NOT invent tasks, actors, systems, times, or volumes not in the text.\n  - Do NOT estimate durations or costs \u2014 that is a later agent's job.\n  - Physical, in-person, or non-digitizable steps must be graded \"manual\".\n  - Steps that only require reading text and producing structured/textual output\n    are typically \"full\"; steps needing human judgment or sign-off are \"partial\".\n\nReturn ONLY a JSON object with this exact shape:\n{{\n  \"reasoning\": \"<your concise step-by-step reasoning>\",\n  \"work_units\": [\n    {{\n      \"id\": \"w1\",\n      \"name\": \"<short task label>\",\n      \"actor\": \"<role/team or {MISSING}>\",\n      \"system\": \"<tool/system or {MISSING}>\",\n      \"description\": \"<one-line paraphrase of the task>\",\n      \"auto_grade\": \"full | partial | manual\",\n      \"auto_rationale\": \"<one sentence: why this grade>\",\n      \"evidence\": \"<short phrase from the transcript this unit came from>\"\n    }}\n  ]\n}}\n\"\"\"\n\nAGENT1_USER = f\"\"\"\\\nHere is the operational interview transcript. Extract and classify all work units.\n\n--- BEGIN TRANSCRIPT ---\n{INTERVIEW_TEXT}\n--- END TRANSCRIPT ---\n\"\"\"\n\nprint(f\"[INFO] Agent-1 prompt ready \"\n      f\"(system {len(AGENT1_SYSTEM)} chars, user {len(AGENT1_USER)} chars).\")\n\n\n# %%\n# =============================================================================\n# Cell 5. Run Agent 1 (deterministic: temperature = 0.0)\n# =============================================================================\nCFG = manifest[\"pipeline_cfg\"][\"agent1_extract\"]  # {tier, temperature, n_samples}\n\nresult = llm_call(\n    system=AGENT1_SYSTEM,\n    user=AGENT1_USER,\n    tier=CFG[\"tier\"],\n    temperature=CFG[\"temperature\"],\n    response_json=True,\n    tag=\"agent1_extract\",\n)\n\npayload = result[\"json\"] or {}\nraw_units = payload.get(\"work_units\", [])\nprint(f\"[INFO] Agent 1 returned {len(raw_units)} work units \"\n      f\"(cached={result['cached']}, cost=${result['cost_usd']:.5f}).\")\nprint(f\"[INFO] Model reasoning (preview): \"\n      f\"{str(payload.get('reasoning',''))[:300]}...\")\n\n\n# %%\n# =============================================================================\n# Cell 6. Normalize into the WorkUnit schema + validate (quality gate DP1)\n#\n# A validation gate between agents localizes errors: we coerce every record to\n# the WorkUnit schema, enforce valid grades, assign stable ids, and flag any\n# unit the model returned in an unexpected shape.\n# =============================================================================\ndef normalize_units(raw: list[dict]) -> tuple[list[WorkUnit], list[str]]:\n    units: list[WorkUnit] = []\n    problems: list[str] = []\n    valid_grades = set(AUTO_GRADES)\n\n    for i, r in enumerate(raw, start=1):\n        if not isinstance(r, dict) or not r.get(\"name\"):\n            problems.append(f\"record {i}: missing/invalid 'name' -> skipped\")\n            continue\n        grade = str(r.get(\"auto_grade\", MISSING)).strip().lower()\n        if grade not in valid_grades:\n            problems.append(\n                f\"record {i} ('{r.get('name')}'): invalid grade \"\n                f\"'{r.get('auto_grade')}' -> set to {MISSING}\"\n            )\n            grade = MISSING\n        units.append(WorkUnit(\n            id=r.get(\"id\") or f\"w{i}\",\n            name=str(r.get(\"name\")).strip(),\n            actor=str(r.get(\"actor\", MISSING)).strip() or MISSING,\n            system=str(r.get(\"system\", MISSING)).strip() or MISSING,\n            description=str(r.get(\"description\", \"\")).strip(),\n            auto_grade=grade,\n            auto_rationale=str(r.get(\"auto_rationale\", \"\")).strip(),\n            evidence=str(r.get(\"evidence\", \"\")).strip(),\n        ))\n    # Re-assign stable sequential ids so downstream agents can rely on them.\n    for k, u in enumerate(units, start=1):\n        u.id = f\"w{k}\"\n    return units, problems\n\n\nwork_units, problems = normalize_units(raw_units)\n\nprint(f\"[INFO] Normalized {len(work_units)} work units.\")\nif problems:\n    print(f\"[WARN] {len(problems)} validation issue(s):\")\n    for p in problems:\n        print(\"   -\", p)\nelse:\n    print(\"[INFO] All records passed the validation gate.\")\n\n\n# %%\n# =============================================================================\n# Cell 7. Inspect results as a table (paper-ready view)\n# =============================================================================\ndf1 = pd.DataFrame([u.to_dict() for u in work_units])\nview_cols = [\"id\", \"name\", \"actor\", \"system\", \"auto_grade\", \"auto_rationale\"]\ndf1_view = df1[view_cols] if not df1.empty else df1\n\nwith pd.option_context(\"display.max_colwidth\", 60, \"display.width\", 160):\n    print(df1_view.to_string(index=False))\n\n# Grade distribution \u2014 a first sanity check on the extraction.\nif not df1.empty:\n    dist = df1[\"auto_grade\"].value_counts().reindex(\n        list(AUTO_GRADES) + [MISSING]).fillna(0).astype(int)\n    print(\"\\n[INFO] Automatability grade distribution:\")\n    for g, n in dist.items():\n        print(f\"   {g:8s}: {n}\")\n\n\n# %%\n# =============================================================================\n# Cell 8. Persist Agent-1 output for the next stage (Agent 2) + cost log\n# =============================================================================\nOUT_PATH = INFER / \"agent1_work_units.json\"\nout_obj = {\n    \"agent\": \"agent1_task_extraction\",\n    \"interview_source\": rel(INTERVIEW_PATH),\n    \"model\": result[\"model\"],\n    \"tier\": CFG[\"tier\"],\n    \"temperature\": CFG[\"temperature\"],\n    \"n_work_units\": len(work_units),\n    \"reasoning\": payload.get(\"reasoning\", \"\"),\n    \"work_units\": [u.to_dict() for u in work_units],\n    \"validation_problems\": problems,\n}\nOUT_PATH.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2),\n                    encoding=\"utf-8\")\nprint(f\"[INFO] Agent-1 output -> {rel(OUT_PATH)}  ({len(work_units)} units)\")\n\n# Also export a CSV table for the paper's appendix.\nif not df1.empty:\n    csv_path = TAB / \"agent1_work_units.csv\"\n    df1[view_cols + [\"evidence\", \"description\"]].to_csv(\n        csv_path, index=False, encoding=\"utf-8-sig\")\n    print(f\"[INFO] Paper table -> {rel(csv_path)}\")\n\nprint(f\"[INFO] Agent-1 spend this run: ${COST.total_usd():.5f}\")\nCOST.flush(\"01_agent1\")   # append to shared cost ledger",
    "015_agent1_5_process_graph": "# %%\n# =============================================================================\n# Cell 3. Load Agent-1 output (the graded work units)\n# =============================================================================\nA1_PATH = INFER / \"agent1_work_units.json\"\nif not A1_PATH.exists():\n    raise FileNotFoundError(\n        \"[ERROR] agent1_work_units.json missing. Run 01_agent1_task_extraction first.\"\n    )\na1 = json.loads(A1_PATH.read_text(encoding=\"utf-8\"))\nunits = a1[\"work_units\"]\nprint(f\"[INFO] Loaded {len(units)} work units from {rel(A1_PATH)}\")\n\n# Ordered list of lanes (organizational actors) as they first appear.\n# Lanes are derived from the data \u2014 no domain-specific lane names are hardcoded.\nlane_order: list[str] = []\nfor u in units:\n    actor = u.get(\"actor\") or MISSING\n    if actor not in lane_order:\n        lane_order.append(actor)\nprint(f\"[INFO] Derived {len(lane_order)} organizational lanes: {lane_order}\")\n\n\n# %%\n# =============================================================================\n# Cell 4. Agent-1.5 prompt \u2014 assign shape TYPE + branch edges (domain-agnostic)\n#\n# The model sees only the ordered work units (id, name, actor, grade) and must:\n#   * assign each unit a flowchart shape type: start | task | decision | end\n#   * declare edges (from_id -> to_id), including a branch label for decisions\n# This mirrors the author's legend (Image 1): start/end hexagons, activity\n# rectangles, decision diamonds, and directional flow. Nothing here references\n# the specific business domain.\n# =============================================================================\ncompact_units = [\n    {\"id\": u[\"id\"], \"name\": u[\"name\"],\n     \"actor\": u.get(\"actor\", MISSING), \"grade\": u.get(\"auto_grade\", MISSING)}\n    for u in units\n]\n\nAGENT15_SYSTEM = \"\"\"\\\nYou are Agent 1.5 (Process-Graph Construction) in an ROI-estimation pipeline.\nYou receive an ORDERED list of work units already extracted from an operational\ninterview (any business domain). Your job is to turn them into a process graph.\n\nFor every unit, assign exactly one flowchart SHAPE TYPE:\n  - \"start\"    : the entry point that triggers the process (usually the first unit)\n  - \"end\"      : a terminal unit that dispatches/closes the process\n  - \"decision\" : a unit that branches on a condition (yes/no, new/re-run, pass/fail)\n  - \"task\"     : any ordinary activity that is neither start, end, nor a branch\n\nThen declare directed EDGES connecting the units in the order the process flows.\n  - Normal flow: {\"from\": \"wX\", \"to\": \"wY\", \"label\": \"\"}\n  - Decision branches: emit two edges from the decision unit, each with a short\n    branch label, e.g. \"yes\"/\"no\" or \"new\"/\"re-run\".\n  - The graph should be a single connected flow from the start unit to the end\n    unit(s). Keep it mostly linear; add branches only where the units imply them.\n\nRules:\n  - Use ONLY the given unit ids. Do not invent units.\n  - Exactly one \"start\". At least one \"end\".\n  - Do not change names or grades; you only add \"type\" and the edge list.\n\nReturn ONLY JSON:\n{\n  \"nodes\": [ {\"id\": \"w1\", \"type\": \"start|task|decision|end\"}, ... ],\n  \"edges\": [ {\"from\": \"wX\", \"to\": \"wY\", \"label\": \"\"}, ... ]\n}\n\"\"\"\n\nAGENT15_USER = (\n    \"Ordered work units (id, name, actor, grade):\\n\"\n    + json.dumps(compact_units, ensure_ascii=False, indent=1)\n)\n\nprint(f\"[INFO] Agent-1.5 prompt ready (system {len(AGENT15_SYSTEM)} chars, \"\n      f\"user {len(AGENT15_USER)} chars).\")\n\n\n# %%\n# =============================================================================\n# Cell 5. Run Agent 1.5 (deterministic) and validate the returned graph\n# =============================================================================\nres = llm_call(\n    system=AGENT15_SYSTEM, user=AGENT15_USER,\n    tier=DEFAULT_TIER, temperature=0.0,\n    response_json=True, tag=\"agent15_graph\",\n)\ngraph = res[\"json\"] or {}\nnodes_raw = graph.get(\"nodes\", [])\nedges_raw = graph.get(\"edges\", [])\nprint(f\"[INFO] Agent 1.5 returned {len(nodes_raw)} nodes, {len(edges_raw)} edges \"\n      f\"(cached={res['cached']}, cost=${res['cost_usd']:.5f}).\")\n\n# --- Validation gate: coerce types, guarantee ids, repair start/end ----------\nVALID_TYPES = {\"start\", \"task\", \"decision\", \"end\"}\nid2unit = {u[\"id\"]: u for u in units}\ntype_by_id: dict[str, str] = {}\n\nfor n in nodes_raw:\n    nid = n.get(\"id\")\n    t = str(n.get(\"type\", \"task\")).strip().lower()\n    if nid in id2unit and t in VALID_TYPES:\n        type_by_id[nid] = t\n\n# Any unit the model omitted defaults to \"task\".\nfor u in units:\n    type_by_id.setdefault(u[\"id\"], \"task\")\n\n# Guarantee exactly one start and at least one end using unit order as fallback.\nordered_ids = [u[\"id\"] for u in units]\nif \"start\" not in type_by_id.values():\n    type_by_id[ordered_ids[0]] = \"start\"\nif \"end\" not in type_by_id.values():\n    type_by_id[ordered_ids[-1]] = \"end\"\n\n# Keep only edges between known ids.\nedges = [\n    {\"from\": e[\"from\"], \"to\": e[\"to\"], \"label\": str(e.get(\"label\", \"\")).strip()}\n    for e in edges_raw\n    if e.get(\"from\") in id2unit and e.get(\"to\") in id2unit\n]\n\n# If the model produced too few edges, fall back to a linear chain so the graph\n# is always connected (a safe, transparent default).\nif len(edges) < len(units) - 1:\n    print(\"[WARN] Sparse edge set; adding a linear backbone as fallback.\")\n    have = {(e[\"from\"], e[\"to\"]) for e in edges}\n    for a, b in zip(ordered_ids, ordered_ids[1:]):\n        if (a, b) not in have:\n            edges.append({\"from\": a, \"to\": b, \"label\": \"\"})\n\nfrom collections import Counter\nprint(f\"[INFO] Node type distribution: {dict(Counter(type_by_id.values()))}\")\nprint(f\"[INFO] Final edge count: {len(edges)}\")\n\n\n# %%\n# =============================================================================\n# Cell 6. Assemble the AS-IS graph (typed nodes + lanes + grades + edges)\n# =============================================================================\ndef build_asis_nodes() -> list[dict]:\n    out = []\n    for u in units:\n        out.append({\n            \"id\": u[\"id\"],\n            \"name\": u[\"name\"],\n            \"lane\": u.get(\"actor\", MISSING),      # organizational swimlane\n            \"type\": type_by_id[u[\"id\"]],           # shape type\n            \"grade\": u.get(\"auto_grade\", MISSING), # automatability grade\n            \"rationale\": u.get(\"auto_rationale\", \"\"),\n        })\n    return out\n\n\nasis_nodes = build_asis_nodes()\n\n# Human-touched activity = any node whose work is performed by a person.\n# Convention: a node is \"AI-performed\" in AS-IS only if the interview already\n# attributes it to a system/GPT; everything else is human in AS-IS.\ndef is_ai_actor(node: dict) -> bool:\n    a = (node.get(\"lane\") or \"\").lower()\n    s = \"\"  # system field not carried into the graph node; lane is the signal\n    return a in {\"system\", \"gpt\", \"ai\", \"ai agent\"} or \"gpt\" in a\n\nasis_human_nodes = [n for n in asis_nodes if not is_ai_actor(n)]\nprint(f\"[INFO] AS-IS total nodes        : {len(asis_nodes)}\")\nprint(f\"[INFO] AS-IS human-touched nodes: {len(asis_human_nodes)}\")\n\n\n# %%\n# =============================================================================\n# Cell 7. Derive the TO-BE graph by applying the automatability rules\n#\n# full    -> node moves to the AI lane; human_effort_factor = 0.0\n# partial -> node splits: an AI-draft node (AI lane) + a human-review node\n#            (original lane) with human_effort_factor = PARTIAL_RETAIN\n# manual  -> unchanged; human_effort_factor = 1.0\n#\n# We record a per-node \"human_effort_factor\" so the effort agent (nb 02) can\n# multiply it by the estimated minutes to get TO-BE human effort directly.\n# =============================================================================\nPARTIAL_RETAIN = 0.30          # residual human effort for partial tasks (parameter)\nAI_LANE = \"AI Agent\"           # single synthetic lane for AI-performed work\n\ndef derive_tobe(asis_nodes: list[dict]) -> tuple[list[dict], list[dict]]:\n    tobe_nodes: list[dict] = []\n    id_map: dict[str, list[str]] = {}   # asis id -> resulting tobe ids (for edges)\n\n    for n in asis_nodes:\n        g = n.get(\"grade\", MISSING)\n        base = {\n            \"src_id\": n[\"id\"], \"name\": n[\"name\"], \"type\": n[\"type\"],\n            \"grade\": g, \"rationale\": n.get(\"rationale\", \"\"),\n        }\n        if g == \"full\":\n            nid = n[\"id\"] + \"_ai\"\n            tobe_nodes.append({**base, \"id\": nid, \"lane\": AI_LANE,\n                               \"human_effort_factor\": 0.0, \"role\": \"ai\"})\n            id_map[n[\"id\"]] = [nid]\n        elif g == \"partial\":\n            ai_id = n[\"id\"] + \"_ai\"\n            hr_id = n[\"id\"] + \"_review\"\n            tobe_nodes.append({**base, \"id\": ai_id, \"lane\": AI_LANE,\n                               \"name\": n[\"name\"] + \" (AI draft)\",\n                               \"human_effort_factor\": 0.0, \"role\": \"ai\"})\n            tobe_nodes.append({**base, \"id\": hr_id, \"lane\": n[\"lane\"],\n                               \"name\": n[\"name\"] + \" (human review)\",\n                               \"human_effort_factor\": PARTIAL_RETAIN, \"role\": \"human\"})\n            id_map[n[\"id\"]] = [ai_id, hr_id]\n        else:  # manual (or MISSING treated conservatively as manual)\n            nid = n[\"id\"] + \"_m\"\n            tobe_nodes.append({**base, \"id\": nid, \"lane\": n[\"lane\"],\n                               \"human_effort_factor\": 1.0, \"role\": \"human\"})\n            id_map[n[\"id\"]] = [nid]\n    return tobe_nodes, id_map\n\n\ndef remap_edges(edges: list[dict], id_map: dict[str, list[str]]) -> list[dict]:\n    \"\"\"Reconnect edges through the TO-BE id expansion. For a split (partial)\n    node, flow enters the AI-draft node and exits the human-review node.\"\"\"\n    def entry(src): return id_map[src][0]          # first resulting node\n    def exit_(src): return id_map[src][-1]          # last resulting node\n    out = []\n    for e in edges:\n        if e[\"from\"] in id_map and e[\"to\"] in id_map:\n            out.append({\"from\": exit_(e[\"from\"]), \"to\": entry(e[\"to\"]),\n                        \"label\": e.get(\"label\", \"\")})\n    # Also stitch the internal AI->review edge for each split node.\n    for src, ids in id_map.items():\n        if len(ids) == 2:\n            out.append({\"from\": ids[0], \"to\": ids[1], \"label\": \"\"})\n    return out\n\n\ntobe_nodes, id_map = derive_tobe(asis_nodes)\ntobe_edges = remap_edges(edges, id_map)\ntobe_human_nodes = [n for n in tobe_nodes if n[\"role\"] == \"human\"]\n\n# --- Three distinct reduction metrics (the paper reports all three) ----------\n# Metric A: raw human-touched NODE COUNT (undercounts partial reductions,\n#           because each 'partial' task retains one human-review node).\n# Metric B: human EFFORT (time-weighted) \u2014 computed in nb 02 once Agent 2 has\n#           estimated minutes; this is the primary ROI metric.\n# Metric C: fully-eliminated human activities (grade 'full' & human in AS-IS)\n#           vs. partially reduced ones \u2014 closest to the author's headline count.\nasis_human_full = [n for n in asis_nodes\n                   if not is_ai_actor(n) and n.get(\"grade\") == \"full\"]\nasis_human_partial = [n for n in asis_nodes\n                      if not is_ai_actor(n) and n.get(\"grade\") == \"partial\"]\n\nprint(f\"[INFO] TO-BE total nodes        : {len(tobe_nodes)}\")\nprint(f\"[INFO] TO-BE human-touched nodes: {len(tobe_human_nodes)}\")\nprint(\"[INFO] --- Reduction metrics ---\")\nprint(f\"[Metric A] Human node count : {len(asis_human_nodes)} -> \"\n      f\"{len(tobe_human_nodes)} \"\n      f\"({(1 - len(tobe_human_nodes)/max(len(asis_human_nodes),1))*100:.0f}% down)\")\nprint(f\"[Metric C] Fully-eliminated human activities: {len(asis_human_full)}; \"\n      f\"partially reduced: {len(asis_human_partial)}\")\nprint(\"[Metric B] Human EFFORT (time-weighted) -> computed in nb 02 (Agent 2).\")\n\n\n# %%\n# =============================================================================\n# Cell 8. Persist the process graph (AS-IS + TO-BE) for effort & figure stages\n# =============================================================================\nGRAPH_PATH = INFER / \"process_graph.json\"\ngraph_obj = {\n    \"agent\": \"agent1_5_process_graph\",\n    \"source\": rel(A1_PATH),\n    \"params\": {\"partial_retain\": PARTIAL_RETAIN, \"ai_lane\": AI_LANE},\n    \"lane_order\": lane_order,\n    \"as_is\": {\"nodes\": asis_nodes, \"edges\": edges},\n    \"to_be\": {\"nodes\": tobe_nodes, \"edges\": tobe_edges},\n    \"counts\": {\n        \"asis_total\": len(asis_nodes),\n        \"asis_human\": len(asis_human_nodes),\n        \"tobe_total\": len(tobe_nodes),\n        \"tobe_human\": len(tobe_human_nodes),\n        # Metric A: human-node-count reduction (%).\n        \"node_count_reduction_pct\": round(\n            (1 - len(tobe_human_nodes) / max(len(asis_human_nodes), 1)) * 100, 1),\n        # Metric C: activities fully vs. partially freed of human effort.\n        \"fully_eliminated_human\": len(asis_human_full),\n        \"partially_reduced_human\": len(asis_human_partial),\n        # Metric B (effort/time reduction %) is filled downstream by Agent 2.\n        \"effort_reduction_pct\": None,\n    },\n}\nGRAPH_PATH.write_text(json.dumps(graph_obj, ensure_ascii=False, indent=2),\n                      encoding=\"utf-8\")\nprint(f\"[INFO] Process graph -> {rel(GRAPH_PATH)}\")\n\n# Paper table: per-node AS-IS/TO-BE mapping with grades and effort factors.\nrows = []\nfor n in asis_nodes:\n    for tid in id_map[n[\"id\"]]:\n        tn = next(t for t in tobe_nodes if t[\"id\"] == tid)\n        rows.append({\n            \"asis_id\": n[\"id\"], \"name\": n[\"name\"], \"lane\": n[\"lane\"],\n            \"type\": n[\"type\"], \"grade\": n[\"grade\"],\n            \"tobe_id\": tn[\"id\"], \"tobe_lane\": tn[\"lane\"],\n            \"role\": tn[\"role\"], \"human_effort_factor\": tn[\"human_effort_factor\"],\n        })\nmap_df = pd.DataFrame(rows)\nmap_csv = TAB / \"asis_tobe_mapping.csv\"\nmap_df.to_csv(map_csv, index=False, encoding=\"utf-8-sig\")\nprint(f\"[INFO] AS-IS/TO-BE mapping table -> {rel(map_csv)}\")\nprint(f\"[INFO] Agent-1.5 spend this run: ${COST.total_usd():.5f}\")",
    "02_agent2_time_estimation": "# %%\n# =============================================================================\n# Cell 3. Load the process graph and the interview text\n# =============================================================================\nGRAPH_PATH = INFER / \"process_graph.json\"\nif not GRAPH_PATH.exists():\n    raise FileNotFoundError(\"[ERROR] process_graph.json missing. Run nb 015 first.\")\nG = json.loads(GRAPH_PATH.read_text(encoding=\"utf-8\"))\nasis_nodes = G[\"as_is\"][\"nodes\"]\nPARTIAL_RETAIN = G[\"params\"][\"partial_retain\"]\n\nINTERVIEW_PATH = DATA_RAW / \"interview_tcb.txt\"\nINTERVIEW_TEXT = INTERVIEW_PATH.read_text(encoding=\"utf-8\").strip()\nprint(f\"[INFO] AS-IS nodes: {len(asis_nodes)}  | interview: {len(INTERVIEW_TEXT)} chars\")\nprint(f\"[INFO] partial_retain (TO-BE residual human effort) = {PARTIAL_RETAIN}\")\n\n\n# %%\n# =============================================================================\n# Cell 4. Estimation configuration \u2014 shape-type priors + qualitative weights\n#\n# These are EDITABLE priors, not measured constants. They encode a transparent\n# default when the interview states no numbers. Every field is a parameter so a\n# user can retune per engagement; nothing is domain-specific.\n# =============================================================================\n# Base minutes-per-case prior by flowchart shape type.\nSHAPE_PRIOR_MIN = {\n    \"start\": 3.0,     # intake / trigger \u2014 short\n    \"task\": 15.0,     # ordinary activity\n    \"decision\": 8.0,  # judgment / branch\n    \"end\": 5.0,       # closeout / dispatch\n}\n\n# Global default monthly case volume (interview states none). Agent 3 will\n# sweep this in sensitivity analysis, so a single transparent default is honest.\nDEFAULT_CASES_PER_MONTH = 20.0\n\n# Qualitative multipliers applied when the interview flags a task as unusually\n# heavy or wasteful. Keys are neutral concepts; the LLM maps nodes to them.\nQUALITATIVE_WEIGHT = {\n    \"most_time_consuming\": 4.0,   # explicitly named as the biggest time sink\n    \"repeated_effort\": 2.0,       # \"a lot of repeated copy-paste / editing\"\n    \"manual_data_entry\": 2.0,     # \"manual entry ... regrettable time sink\"\n    \"normal\": 1.0,                # default\n}\n\n# Self-Consistency settings (Reliability axis).\nN_ROLLOUTS = manifest[\"pipeline_cfg\"][\"agent2_time\"][\"n_samples\"]   # e.g. 5\nSC_TEMPERATURE = manifest[\"pipeline_cfg\"][\"agent2_time\"][\"temperature\"]  # e.g. 0.7\n\nprint(f\"[INFO] Shape priors (min): {SHAPE_PRIOR_MIN}\")\nprint(f\"[INFO] Default cases/month: {DEFAULT_CASES_PER_MONTH}\")\nprint(f\"[INFO] Self-Consistency: N={N_ROLLOUTS} rollouts @ temp={SC_TEMPERATURE}\")\n\n\n# %%\n# =============================================================================\n# Cell 5. Agent-2 prompt \u2014 per-node time estimate with source tagging\n#\n# The LLM does NOT compute monthly totals (that is code, DP2). It only proposes,\n# per node: minutes_per_case, cases_per_month, a qualitative weight key, a\n# SOURCE tag, a confidence in [0,1], and a one-line rationale. The prompt is\n# domain-agnostic; it reasons from the transcript plus the shape-type context.\n# =============================================================================\nnode_ctx = [\n    {\"id\": n[\"id\"], \"name\": n[\"name\"], \"type\": n[\"type\"],\n     \"actor\": n.get(\"lane\", MISSING), \"grade\": n.get(\"grade\", MISSING)}\n    for n in asis_nodes\n]\n\nAGENT2_SYSTEM = f\"\"\"\\\nYou are Agent 2 (Work-Time Estimation) in an ROI-estimation pipeline. You are\ngiven an operational interview transcript (any domain) and a list of process\nnodes already extracted from it. For EACH node, estimate how long the task takes\nand how often it runs \u2014 but be explicit about how sure you are.\n\nFor each node output:\n  - \"minutes_per_case\": a positive number \u2014 time for ONE occurrence of the task.\n  - \"cases_per_month\": a positive number \u2014 how many times per month it runs.\n  - \"weight_key\": one of {list(QUALITATIVE_WEIGHT)} \u2014 pick \"most_time_consuming\",\n     \"repeated_effort\", or \"manual_data_entry\" ONLY if the transcript clearly\n     signals it for this node; otherwise \"normal\".\n  - \"source\": one of:\n       \"stated\"  \u2014 the transcript gives an explicit number for this task,\n       \"implied\" \u2014 no number, but the transcript qualitatively signals its size,\n       \"prior\"   \u2014 neither; you are relying on a generic default for its type.\n  - \"confidence\": a number in [0,1] reflecting how well-grounded the estimate is\n     (stated ~0.9, implied ~0.5, prior ~0.3).\n  - \"rationale\": one sentence citing the transcript phrase or the default basis.\n\nRules:\n  - Do NOT compute monthly totals; only per-case time and monthly frequency.\n  - Do NOT fabricate explicit numbers. If the transcript has none for a node,\n    the source MUST be \"implied\" or \"prior\", never \"stated\".\n  - Keep estimates realistic and internally consistent across similar tasks.\n\nReturn ONLY JSON:\n{{ \"estimates\": [\n     {{ \"id\": \"wX\", \"minutes_per_case\": <num>, \"cases_per_month\": <num>,\n        \"weight_key\": \"...\", \"source\": \"stated|implied|prior\",\n        \"confidence\": <0..1>, \"rationale\": \"...\" }}\n] }}\n\"\"\"\n\ndef build_user(nodes_ctx: list[dict]) -> str:\n    return (\n        \"--- INTERVIEW TRANSCRIPT ---\\n\" + INTERVIEW_TEXT +\n        \"\\n--- END TRANSCRIPT ---\\n\\nProcess nodes to estimate:\\n\" +\n        json.dumps(nodes_ctx, ensure_ascii=False, indent=1)\n    )\n\nAGENT2_USER = build_user(node_ctx)\nprint(f\"[INFO] Agent-2 prompt ready (system {len(AGENT2_SYSTEM)} chars, \"\n      f\"user {len(AGENT2_USER)} chars).\")\n\n\n# %%\n# =============================================================================\n# Cell 6. Self-Consistency: run N stochastic rollouts, collect per-node samples\n# =============================================================================\ndef run_rollouts(n: int) -> list[dict]:\n    \"\"\"Return a list of {id -> estimate dict} maps, one per rollout.\"\"\"\n    maps = []\n    for k in range(n):\n        r = llm_call(\n            system=AGENT2_SYSTEM, user=AGENT2_USER,\n            tier=DEFAULT_TIER, temperature=SC_TEMPERATURE,\n            response_json=True, tag=f\"agent2_rollout_{k}\",\n            use_cache=True, salt=f\"rollout-{k}\",   # distinct cache per rollout\n        )\n        est = (r[\"json\"] or {}).get(\"estimates\", [])\n        m = {e[\"id\"]: e for e in est if isinstance(e, dict) and e.get(\"id\")}\n        maps.append(m)\n        print(f\"   rollout {k+1}/{n}: {len(m)} estimates \"\n              f\"(cached={r['cached']}, cost=${r['cost_usd']:.5f})\")\n    return maps\n\n\nprint(f\"[INFO] Running {N_ROLLOUTS} Self-Consistency rollouts ...\")\nrollout_maps = run_rollouts(N_ROLLOUTS)\n\n\n# %%\n# =============================================================================\n# Cell 7. Reduce rollouts -> median estimate + dispersion (Reliability signal)\n#\n# For each node we take the MEDIAN of minutes_per_case and cases_per_month\n# across rollouts (robust to outliers), and record the coefficient of variation\n# (CV = std/mean) as a per-node reliability signal. The most common source tag\n# and the mean confidence are carried through.\n# =============================================================================\ndef _nums(maps, nid, field):\n    vals = []\n    for m in maps:\n        v = m.get(nid, {}).get(field)\n        if isinstance(v, (int, float)) and v > 0:\n            vals.append(float(v))\n    return vals\n\ndef _cv(vals):\n    if len(vals) < 2:\n        return 0.0\n    mu = statistics.mean(vals)\n    return (statistics.pstdev(vals) / mu) if mu else 0.0\n\nreduced = []\nfor n in asis_nodes:\n    nid = n[\"id\"]\n    mpc_vals = _nums(rollout_maps, nid, \"minutes_per_case\")\n    cpm_vals = _nums(rollout_maps, nid, \"cases_per_month\")\n\n    # Fallbacks to shape-type prior / global default when a rollout omitted a node.\n    mpc = statistics.median(mpc_vals) if mpc_vals else SHAPE_PRIOR_MIN.get(n[\"type\"], 15.0)\n    cpm = statistics.median(cpm_vals) if cpm_vals else DEFAULT_CASES_PER_MONTH\n\n    # Source + confidence: majority source, mean confidence across rollouts.\n    srcs = [rollout_maps[i].get(nid, {}).get(\"source\", \"prior\")\n            for i in range(len(rollout_maps)) if nid in rollout_maps[i]]\n    source = max(set(srcs), key=srcs.count) if srcs else \"prior\"\n    confs = [rollout_maps[i].get(nid, {}).get(\"confidence\", 0.3)\n             for i in range(len(rollout_maps)) if nid in rollout_maps[i]]\n    confidence = round(float(statistics.mean(confs)), 3) if confs else 0.3\n    wkeys = [rollout_maps[i].get(nid, {}).get(\"weight_key\", \"normal\")\n             for i in range(len(rollout_maps)) if nid in rollout_maps[i]]\n    weight_key = max(set(wkeys), key=wkeys.count) if wkeys else \"normal\"\n\n    reduced.append({\n        \"id\": nid, \"name\": n[\"name\"], \"type\": n[\"type\"],\n        \"lane\": n.get(\"lane\", MISSING), \"grade\": n.get(\"grade\", MISSING),\n        \"minutes_per_case\": round(mpc, 2),\n        \"cases_per_month\": round(cpm, 2),\n        \"weight_key\": weight_key,\n        \"source\": source, \"confidence\": confidence,\n        \"cv_minutes\": round(_cv(mpc_vals), 3),   # reliability signal\n        \"n_samples\": len(mpc_vals),\n    })\n\nprint(f\"[INFO] Reduced {len(reduced)} node estimates across \"\n      f\"{N_ROLLOUTS} rollouts.\")\n\n\n# %%\n# =============================================================================\n# Cell 8. Compute effort IN CODE (DP2): AS-IS and TO-BE monthly minutes\n#\n# monthly_minutes = minutes_per_case * cases_per_month * qualitative_weight\n# AS-IS human effort: nodes NOT already AI-performed contribute full effort.\n# TO-BE human effort: multiply by the per-node human_effort_factor implied by\n#   its grade (full->0, partial->PARTIAL_RETAIN, manual->1).\n# =============================================================================\nAI_LANES = {\"system\", \"gpt\", \"ai\", \"ai agent\"}\n\ndef is_ai(lane: str) -> bool:\n    l = (lane or \"\").lower()\n    return l in AI_LANES or \"gpt\" in l\n\ndef human_factor(grade: str) -> float:\n    return {\"full\": 0.0, \"partial\": PARTIAL_RETAIN, \"manual\": 1.0}.get(grade, 1.0)\n\nrows = []\nasis_human_min = 0.0\ntobe_human_min = 0.0\nfor e in reduced:\n    w = QUALITATIVE_WEIGHT.get(e[\"weight_key\"], 1.0)\n    monthly = e[\"minutes_per_case\"] * e[\"cases_per_month\"] * w   # DP2: code, not LLM\n\n    asis_is_human = not is_ai(e[\"lane\"])\n    asis_min = monthly if asis_is_human else 0.0\n    # TO-BE: if AS-IS was human, apply the grade's human factor; AI stays 0.\n    tobe_min = (monthly * human_factor(e[\"grade\"])) if asis_is_human else 0.0\n\n    asis_human_min += asis_min\n    tobe_human_min += tobe_min\n    rows.append({**e, \"qual_weight\": w,\n                 \"monthly_minutes\": round(monthly, 1),\n                 \"asis_human_minutes\": round(asis_min, 1),\n                 \"tobe_human_minutes\": round(tobe_min, 1)})\n\ndf2 = pd.DataFrame(rows)\neffort_reduction_pct = (1 - tobe_human_min / asis_human_min) * 100 if asis_human_min else 0.0\n\nprint(f\"[INFO] AS-IS human effort : {asis_human_min:,.0f} min/month \"\n      f\"({asis_human_min/60:,.1f} h)\")\nprint(f\"[INFO] TO-BE human effort : {tobe_human_min:,.0f} min/month \"\n      f\"({tobe_human_min/60:,.1f} h)\")\nprint(f\"[Metric B] Human-effort reduction: {effort_reduction_pct:.1f}%  \"\n      f\"<-- primary ROI input\")\n\n\n# %%\n# =============================================================================\n# Cell 9. Robustness: clarifying questions for low-confidence, high-impact nodes\n#\n# DP3 in action: rather than silently trusting weak estimates, flag the nodes\n# where a human should confirm \u2014 those with low confidence AND large effort.\n# These questions are an artifact of the Robustness axis, not a failure.\n# =============================================================================\nCONF_THRESHOLD = 0.4\nimpact = df2[\"asis_human_minutes\"].fillna(0)\nflagged = df2[(df2[\"confidence\"] < CONF_THRESHOLD) & (impact > impact.median())]\nclarifying = []\nfor _, r in flagged.iterrows():\n    clarifying.append({\n        \"id\": r[\"id\"], \"name\": r[\"name\"],\n        \"why\": f\"low confidence ({r['confidence']}) but high effort \"\n               f\"({r['asis_human_minutes']:.0f} min/mo)\",\n        \"question\": f\"For '{r['name']}', what is the actual time per case and \"\n                    f\"the monthly volume? (current estimate is a \"\n                    f\"{r['source']}-based guess)\",\n    })\nprint(f\"[INFO] {len(clarifying)} clarifying question(s) generated \"\n      f\"(low-confidence & high-impact nodes).\")\nfor c in clarifying[:5]:\n    print(\"   -\", c[\"question\"])\n\n\n# %%\n# =============================================================================\n# Cell 10. Persist Agent-2 output + write Metric B back to the process graph\n# =============================================================================\nOUT_PATH = INFER / \"agent2_time.json\"\nout_obj = {\n    \"agent\": \"agent2_time_estimation\",\n    \"n_rollouts\": N_ROLLOUTS,\n    \"sc_temperature\": SC_TEMPERATURE,\n    \"params\": {\n        \"shape_prior_min\": SHAPE_PRIOR_MIN,\n        \"default_cases_per_month\": DEFAULT_CASES_PER_MONTH,\n        \"qualitative_weight\": QUALITATIVE_WEIGHT,\n        \"partial_retain\": PARTIAL_RETAIN,\n    },\n    \"totals\": {\n        \"asis_human_minutes_per_month\": round(asis_human_min, 1),\n        \"tobe_human_minutes_per_month\": round(tobe_human_min, 1),\n        \"effort_reduction_pct\": round(effort_reduction_pct, 1),\n    },\n    \"estimates\": rows,\n    \"clarifying_questions\": clarifying,\n}\nOUT_PATH.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2),\n                    encoding=\"utf-8\")\nprint(f\"[INFO] Agent-2 output -> {rel(OUT_PATH)}\")\n\n# Write Metric B back into process_graph.json so nb 03 (ROI) and the figure\n# stage can read a single consistent set of counts.\nG[\"counts\"][\"effort_reduction_pct\"] = round(effort_reduction_pct, 1)\nG[\"counts\"][\"asis_human_minutes_per_month\"] = round(asis_human_min, 1)\nG[\"counts\"][\"tobe_human_minutes_per_month\"] = round(tobe_human_min, 1)\nGRAPH_PATH.write_text(json.dumps(G, ensure_ascii=False, indent=2), encoding=\"utf-8\")\nprint(f\"[INFO] Metric B written back to {rel(GRAPH_PATH)}\")\n\n# Paper table: full per-node estimate table with source tags + reliability CV.\ncsv_path = TAB / \"agent2_time_estimates.csv\"\ndf2.to_csv(csv_path, index=False, encoding=\"utf-8-sig\")\nprint(f\"[INFO] Paper table -> {rel(csv_path)}\")\n\n# Reliability summary: distribution of source tags + mean CV.\nsrc_dist = df2[\"source\"].value_counts().to_dict()\nprint(f\"[INFO] Source-tag distribution: {src_dist}\")\nprint(f\"[INFO] Mean reliability CV (minutes): {df2['cv_minutes'].mean():.3f}\")\nprint(f\"[INFO] Agent-2 spend this run: ${COST.total_usd():.5f}\")",
    "03_agent3_roi_computation": "# %%\n# =============================================================================\n# Cell 3. Load Agent-2 estimates + the cost model\n# =============================================================================\nA2_PATH = INFER / \"agent2_time.json\"\nif not A2_PATH.exists():\n    raise FileNotFoundError(\"[ERROR] agent2_time.json missing. Run nb 02 first.\")\nA2 = json.loads(A2_PATH.read_text(encoding=\"utf-8\"))\nestimates = A2[\"estimates\"]\nbase_partial_retain = A2[\"params\"][\"partial_retain\"]\ndefault_cpm = A2[\"params\"][\"default_cases_per_month\"]\n\nW_BASE     = float(COST_MODEL[\"hourly_wage_usd\"])\nCAPEX      = float(COST_MODEL[\"capex_usd\"])\nOPEX_YEAR  = float(COST_MODEL[\"opex_usd_per_year\"])\n\nprint(f\"[INFO] Loaded {len(estimates)} node estimates from {rel(A2_PATH)}\")\nprint(f\"[INFO] Cost model: W=${W_BASE}/h  capex=${CAPEX:,.0f}  opex=${OPEX_YEAR:,.0f}/yr\")\nprint(f\"[INFO] Base partial_retain={base_partial_retain}, \"\n      f\"default cases/month={default_cpm}\")\n\n\n# %%\n# =============================================================================\n# Cell 4. Core ROI engine \u2014 ALL arithmetic in code (DP2)\n#\n# We recompute AS-IS/TO-BE effort from per-node primitives so the engine can be\n# re-evaluated under swept parameters. A node's monthly minutes scale linearly\n# with cases_per_month, so we separate the per-case component from volume:\n#\n#   node_monthly_minutes(cpm) = minutes_per_case * cpm_scaled * qual_weight\n# where cpm_scaled lets us override the (uncertain) volume globally in sweeps.\n# =============================================================================\nAI_LANES = {\"system\", \"gpt\", \"ai\", \"ai agent\"}\n\ndef is_ai(lane: str) -> bool:\n    l = (lane or \"\").lower()\n    return l in AI_LANES or \"gpt\" in l\n\ndef human_factor(grade: str, partial_retain: float) -> float:\n    return {\"full\": 0.0, \"partial\": partial_retain, \"manual\": 1.0}.get(grade, 1.0)\n\n\ndef compute_effort(estimates: list[dict],\n                   partial_retain: float,\n                   cases_per_month_override: Optional[float] = None\n                   ) -> tuple[float, float]:\n    \"\"\"Return (asis_human_minutes_per_month, tobe_human_minutes_per_month).\n\n    If cases_per_month_override is given, EVERY node's volume is replaced by it\n    (used by the sensitivity sweep, since the interview states no volumes). When\n    None, each node keeps its own estimated cases_per_month.\n    \"\"\"\n    asis_min = 0.0\n    tobe_min = 0.0\n    for e in estimates:\n        cpm = cases_per_month_override if cases_per_month_override is not None \\\n            else float(e[\"cases_per_month\"])\n        monthly = float(e[\"minutes_per_case\"]) * cpm * float(e.get(\"qual_weight\", 1.0))\n        if is_ai(e[\"lane\"]):\n            continue  # already AI in AS-IS -> contributes 0 human effort both sides\n        asis_min += monthly\n        tobe_min += monthly * human_factor(e[\"grade\"], partial_retain)\n    return asis_min, tobe_min\n\n\ndef compute_roi(asis_min: float, tobe_min: float,\n                hourly_wage: float, capex: float, opex_year: float) -> dict:\n    \"\"\"Eq. 1 / Eq. 2, in code. Returns saving, ROI, payback, and reduction.\"\"\"\n    saved_min_per_month = max(asis_min - tobe_min, 0.0)\n    saved_hours_per_year = saved_min_per_month * 12.0 / 60.0\n    S = saved_hours_per_year * hourly_wage                     # Eq. 2 (annual saving)\n    net_annual = S - opex_year\n    roi_pct = (net_annual / capex * 100.0) if capex > 0 else float(\"nan\")  # Eq. 1\n    payback_years = (capex / net_annual) if net_annual > 0 else float(\"inf\")\n    reduction_pct = (1 - tobe_min / asis_min) * 100.0 if asis_min > 0 else 0.0\n    return {\n        \"asis_min_month\": asis_min,\n        \"tobe_min_month\": tobe_min,\n        \"saved_hours_year\": saved_hours_per_year,\n        \"annual_saving_usd\": S,\n        \"net_annual_usd\": net_annual,\n        \"roi_pct\": roi_pct,\n        \"payback_years\": payback_years,\n        \"effort_reduction_pct\": reduction_pct,\n    }\n\n\n# --- Baseline ROI (each node keeps its own estimated volume) -----------------\nasis0, tobe0 = compute_effort(estimates, base_partial_retain,\n                              cases_per_month_override=None)\nbase = compute_roi(asis0, tobe0, W_BASE, CAPEX, OPEX_YEAR)\n\nprint(\"[INFO] --- Baseline ROI (code-computed) ---\")\nprint(f\"   AS-IS effort      : {base['asis_min_month']:,.0f} min/mo \"\n      f\"({base['asis_min_month']/60:,.0f} h)\")\nprint(f\"   TO-BE effort      : {base['tobe_min_month']:,.0f} min/mo \"\n      f\"({base['tobe_min_month']/60:,.0f} h)\")\nprint(f\"   Effort reduction  : {base['effort_reduction_pct']:.1f}%\")\nprint(f\"   Saved hours/year  : {base['saved_hours_year']:,.0f} h\")\nprint(f\"   Annual saving (S) : ${base['annual_saving_usd']:,.0f}\")\nprint(f\"   ROI               : {base['roi_pct']:,.0f}%\")\nprint(f\"   Payback           : {base['payback_years']:.2f} years\")\n\n\n# %%\n# =============================================================================\n# Cell 5. Sensitivity analysis \u2014 sweep the 3 uncertain parameters (code)\n#\n# The interview provides no volumes, so cases_per_month is the dominant\n# uncertainty. We sweep it jointly with partial_retain and hourly_wage to give\n# an honest ROI RANGE and a break-even surface, rather than a single number.\n# =============================================================================\nSWEEP = {\n    \"cases_per_month\": [5, 10, 20, 40, 60, 100],   # dominant uncertainty\n    \"partial_retain\":  [0.10, 0.30, 0.50],          # TO-BE assumption\n    \"hourly_wage_usd\": [20, 35, 50, 80],            # region/role\n}\n\nrows = []\nfor cpm, pr, w in itertools.product(SWEEP[\"cases_per_month\"],\n                                    SWEEP[\"partial_retain\"],\n                                    SWEEP[\"hourly_wage_usd\"]):\n    a, t = compute_effort(estimates, pr, cases_per_month_override=cpm)\n    r = compute_roi(a, t, w, CAPEX, OPEX_YEAR)\n    rows.append({\n        \"cases_per_month\": cpm, \"partial_retain\": pr, \"hourly_wage_usd\": w,\n        \"annual_saving_usd\": round(r[\"annual_saving_usd\"], 0),\n        \"roi_pct\": round(r[\"roi_pct\"], 1),\n        \"payback_years\": round(r[\"payback_years\"], 2)\n        if np.isfinite(r[\"payback_years\"]) else None,\n        \"effort_reduction_pct\": round(r[\"effort_reduction_pct\"], 1),\n    })\n\nsens_df = pd.DataFrame(rows)\nprint(f\"[INFO] Sensitivity grid: {len(sens_df)} scenarios \"\n      f\"({len(SWEEP['cases_per_month'])}\u00d7{len(SWEEP['partial_retain'])}\"\n      f\"\u00d7{len(SWEEP['hourly_wage_usd'])}).\")\n\n# ROI range and break-even summary.\nfinite = sens_df.dropna(subset=[\"roi_pct\"])\nprint(f\"[INFO] ROI range across scenarios: \"\n      f\"{finite['roi_pct'].min():,.0f}% to {finite['roi_pct'].max():,.0f}%\")\nbe = sens_df[(sens_df[\"payback_years\"].notna()) & (sens_df[\"payback_years\"] <= 1.0)]\nprint(f\"[INFO] Scenarios with <=1yr payback: {len(be)}/{len(sens_df)}\")\n\n# Break-even volume at the base wage & partial_retain: smallest cpm with ROI>0.\nbe_base = sens_df[(sens_df[\"hourly_wage_usd\"] == W_BASE) &\n                  (sens_df[\"partial_retain\"] == base_partial_retain) &\n                  (sens_df[\"roi_pct\"] > 0)].sort_values(\"cases_per_month\")\nif not be_base.empty:\n    print(f\"[INFO] Break-even volume @ base params: \"\n          f\"cases/month >= {be_base.iloc[0]['cases_per_month']:.0f}\")\n\n\n# %%\n# =============================================================================\n# Cell 6. ReAct interpretation \u2014 LLM writes the executive summary ONLY\n#\n# The model receives the code-computed numbers and produces a concise, honest\n# managerial paragraph. It is explicitly forbidden from introducing new numbers\n# or recomputing anything (DP2). This is the 'reasoning' half of ReAct; the\n# 'action' half (lookup + compute) was done in code above.\n# =============================================================================\nsummary_payload = {\n    \"baseline\": {\n        \"effort_reduction_pct\": round(base[\"effort_reduction_pct\"], 1),\n        \"saved_hours_year\": round(base[\"saved_hours_year\"], 0),\n        \"annual_saving_usd\": round(base[\"annual_saving_usd\"], 0),\n        \"roi_pct\": round(base[\"roi_pct\"], 0),\n        \"payback_years\": round(base[\"payback_years\"], 2),\n    },\n    \"sensitivity\": {\n        \"roi_min_pct\": round(float(finite[\"roi_pct\"].min()), 0),\n        \"roi_max_pct\": round(float(finite[\"roi_pct\"].max()), 0),\n        \"break_even_cases_per_month\":\n            int(be_base.iloc[0][\"cases_per_month\"]) if not be_base.empty else None,\n    },\n    \"cost_model\": {\"capex_usd\": CAPEX, \"opex_usd_per_year\": OPEX_YEAR,\n                   \"hourly_wage_usd\": W_BASE},\n    \"caveat\": \"All figures derive from prior/implied time estimates; the \"\n              \"interview stated no explicit durations or volumes.\",\n}\n\nREACT_SYSTEM = \"\"\"\\\nYou are the reporting layer of an ROI-analysis pipeline. You are given\nALREADY-COMPUTED figures (a baseline scenario and a sensitivity range). Write a\nsingle, sober executive paragraph (4-6 sentences) that a manager could read.\n\nStrict rules:\n  - Use ONLY the numbers provided. Do NOT invent, recompute, or round away any\n    figure. Do NOT add numbers that are not in the input.\n  - State the baseline ROI and payback, then the sensitivity range, then the\n    key caveat about estimate provenance honestly.\n  - No hype. Present the result as conditional on the stated assumptions.\n\nReturn ONLY JSON: {\"executive_summary\": \"<one paragraph>\"}\n\"\"\"\n\nREACT_USER = \"Computed figures:\\n\" + json.dumps(summary_payload, ensure_ascii=False, indent=1)\n\nreact = llm_call(system=REACT_SYSTEM, user=REACT_USER,\n                 tier=DEFAULT_TIER, temperature=0.0,\n                 response_json=True, tag=\"agent3_react_summary\")\nexec_summary = (react[\"json\"] or {}).get(\"executive_summary\", \"\").strip()\nprint(\"[INFO] Executive summary (LLM, interpretation only):\")\nprint(exec_summary)\n\n\n# %%\n# =============================================================================\n# Cell 7. Persist Agent-3 output + paper tables\n# =============================================================================\nOUT_PATH = INFER / \"agent3_roi.json\"\nout_obj = {\n    \"agent\": \"agent3_roi_computation\",\n    \"cost_model\": {\"hourly_wage_usd\": W_BASE, \"capex_usd\": CAPEX,\n                   \"opex_usd_per_year\": OPEX_YEAR},\n    \"baseline\": base,\n    \"sensitivity_sweep\": SWEEP,\n    \"sensitivity_grid\": rows,\n    \"executive_summary\": exec_summary,\n    \"provenance_caveat\": summary_payload[\"caveat\"],\n}\n# Convert any inf payback to a JSON-safe marker.\ndef _json_safe(o):\n    if isinstance(o, float) and not np.isfinite(o):\n        return None\n    if isinstance(o, dict):\n        return {k: _json_safe(v) for k, v in o.items()}\n    if isinstance(o, list):\n        return [_json_safe(v) for v in o]\n    return o\n\nOUT_PATH.write_text(json.dumps(_json_safe(out_obj), ensure_ascii=False, indent=2),\n                    encoding=\"utf-8\")\nprint(f\"[INFO] Agent-3 output -> {rel(OUT_PATH)}\")\n\n# Sensitivity table for the paper appendix.\nsens_csv = TAB / \"agent3_sensitivity.csv\"\nsens_df.to_csv(sens_csv, index=False, encoding=\"utf-8-sig\")\nprint(f\"[INFO] Sensitivity table -> {rel(sens_csv)}\")\n\n# A compact baseline table too.\nbase_row = pd.DataFrame([{\n    \"effort_reduction_pct\": round(base[\"effort_reduction_pct\"], 1),\n    \"saved_hours_year\": round(base[\"saved_hours_year\"], 0),\n    \"annual_saving_usd\": round(base[\"annual_saving_usd\"], 0),\n    \"roi_pct\": round(base[\"roi_pct\"], 0),\n    \"payback_years\": round(base[\"payback_years\"], 2),\n}])\nbase_csv = TAB / \"agent3_baseline_roi.csv\"\nbase_row.to_csv(base_csv, index=False, encoding=\"utf-8-sig\")\nprint(f\"[INFO] Baseline ROI table -> {rel(base_csv)}\")\nprint(f\"[INFO] Agent-3 spend this run: ${COST.total_usd():.5f}\")",
    "04_five_axis_validation": "# %%\n# =============================================================================\n# Cell 3. Load all pipeline artifacts + the partial ground truth\n# =============================================================================\nA1 = json.loads((INFER / \"agent1_work_units.json\").read_text(encoding=\"utf-8\"))\nA2 = json.loads((INFER / \"agent2_time.json\").read_text(encoding=\"utf-8\"))\nA3 = json.loads((INFER / \"agent3_roi.json\").read_text(encoding=\"utf-8\"))\n\nGT_PATH = DATA_RAW / \"ground_truth_tcb.json\"\nGROUND_TRUTH = json.loads(GT_PATH.read_text(encoding=\"utf-8\")) if GT_PATH.exists() else None\n\nunits = A1[\"work_units\"]\nestimates = {e[\"id\"]: e for e in A2[\"estimates\"]}\nprint(f\"[INFO] Agent-1 units: {len(units)}, Agent-2 estimates: {len(estimates)}\")\nprint(f\"[INFO] Ground truth: \"\n      f\"{len(GROUND_TRUTH) if GROUND_TRUTH else 0} expert activities \"\n      f\"{'(Accuracy axis ENABLED)' if GROUND_TRUTH else '(Accuracy axis SKIPPED)'}\")\n\n\n# %%\n# =============================================================================\n# Cell 4. AXIS 1 \u2014 ACCURACY (against the partial expert reference)\n#\n# Challenge: the reference covers only 13 activities and uses different wording\n# than the agent's 46 extracted units. We therefore first ask the LLM to MATCH\n# each ground-truth activity to at most one extracted unit by meaning, then\n# compute agreement ONLY over confident matches. Unmatched reference items are\n# reported as recall gaps \u2014 an honest, explicit limitation, not hidden.\n# =============================================================================\ndef build_matcher_payload():\n    gt = [{\"gt_id\": g[\"id\"], \"name\": g[\"name\"]} for g in GROUND_TRUTH]\n    ex = [{\"unit_id\": u[\"id\"], \"name\": u[\"name\"]} for u in units]\n    return gt, ex\n\naccuracy = {\"enabled\": bool(GROUND_TRUTH)}\nif GROUND_TRUTH:\n    gt_list, ex_list = build_matcher_payload()\n\n    MATCH_SYSTEM = \"\"\"\\\nYou align expert-reference activities to pipeline-extracted work units by MEANING\n(not by exact wording). For each reference activity, choose the single best\nmatching extracted unit, or null if none clearly corresponds.\n\nReturn ONLY JSON:\n{ \"matches\": [ {\"gt_id\": \"...\", \"unit_id\": \"... or null\", \"confidence\": 0..1} ] }\nRules: each unit_id used at most once; use null when no unit is a clear match.\n\"\"\"\n    MATCH_USER = (\"Reference activities:\\n\" + json.dumps(gt_list, ensure_ascii=False)\n                  + \"\\n\\nExtracted units:\\n\" + json.dumps(ex_list, ensure_ascii=False))\n\n    mres = llm_call(MATCH_SYSTEM, MATCH_USER, tier=_MATCHER_TIER, temperature=0.0,\n                    tag=\"axis1_matcher\")\n    matches = (mres[\"json\"] or {}).get(\"matches\", [])\n    # Keep confident, non-null matches.\n    good = [m for m in matches\n            if m.get(\"unit_id\") and m.get(\"confidence\", 0) >= 0.5]\n    gt_by_id = {g[\"id\"]: g for g in GROUND_TRUTH}\n    unit_by_id = {u[\"id\"]: u for u in units}\n    est_by_id = estimates\n\n    # --- Grade agreement + Cohen's kappa over matched pairs ------------------\n    y_true, y_pred, mins_true, mins_pred = [], [], [], []\n    pair_rows = []\n    for m in good:\n        g = gt_by_id.get(m[\"gt_id\"]); u = unit_by_id.get(m[\"unit_id\"])\n        if not g or not u:\n            continue\n        y_true.append(g[\"auto\"]); y_pred.append(u.get(\"auto_grade\", MISSING))\n        e = est_by_id.get(u[\"id\"], {})\n        mpc = e.get(\"minutes_per_case\")\n        if isinstance(mpc, (int, float)) and g.get(\"min\"):\n            mins_true.append(float(g[\"min\"])); mins_pred.append(float(mpc))\n        pair_rows.append({\n            \"gt_id\": g[\"id\"], \"gt_name\": g[\"name\"], \"unit_id\": u[\"id\"],\n            \"gt_grade\": g[\"auto\"], \"pred_grade\": u.get(\"auto_grade\"),\n            \"gt_min\": g.get(\"min\"), \"pred_min\": mpc,\n        })\n\n    def cohen_kappa(a, b):\n        cats = sorted(set(a) | set(b))\n        idx = {c: i for i, c in enumerate(cats)}\n        n = len(a)\n        if n == 0:\n            return None\n        po = sum(1 for x, y in zip(a, b) if x == y) / n\n        # Expected agreement.\n        ca = [0] * len(cats); cb = [0] * len(cats)\n        for x in a: ca[idx[x]] += 1\n        for y in b: cb[idx[y]] += 1\n        pe = sum((ca[i]/n) * (cb[i]/n) for i in range(len(cats)))\n        return (po - pe) / (1 - pe) if (1 - pe) else None\n\n    grade_agree = (sum(1 for x, y in zip(y_true, y_pred) if x == y) / len(y_true)\n                   if y_true else None)\n    kappa = cohen_kappa(y_true, y_pred) if y_true else None\n\n    def mae(a, b):  return float(np.mean(np.abs(np.array(a) - np.array(b)))) if a else None\n    def mape(a, b):\n        a, b = np.array(a, float), np.array(b, float)\n        return float(np.mean(np.abs((a - b) / a)) * 100) if len(a) else None\n\n    accuracy.update({\n        \"matched_pairs\": len(good),\n        \"unmatched_reference\": len(GROUND_TRUTH) - len(good),  # recall gap\n        \"grade_agreement\": round(grade_agree, 3) if grade_agree is not None else None,\n        \"grade_cohen_kappa\": round(kappa, 3) if kappa is not None else None,\n        \"time_MAE_min\": round(mae(mins_true, mins_pred), 2) if mins_true else None,\n        \"time_MAPE_pct\": round(mape(mins_true, mins_pred), 1) if mins_true else None,\n        \"n_time_pairs\": len(mins_true),\n    })\n    print(\"[AXIS 1 \u00b7 Accuracy]\")\n    print(f\"   matched pairs        : {accuracy['matched_pairs']} \"\n          f\"(unmatched ref: {accuracy['unmatched_reference']})\")\n    print(f\"   grade agreement      : {accuracy['grade_agreement']}\")\n    print(f\"   grade Cohen's kappa  : {accuracy['grade_cohen_kappa']}\")\n    print(f\"   time MAE (min)       : {accuracy['time_MAE_min']}\")\n    print(f\"   time MAPE (%)        : {accuracy['time_MAPE_pct']}  \"\n          f\"(n={accuracy['n_time_pairs']})\")\nelse:\n    print(\"[AXIS 1 \u00b7 Accuracy] SKIPPED (no ground truth).\")\n\n\n# %%\n# =============================================================================\n# Cell 5. AXIS 2 \u2014 RELIABILITY (Self-Consistency dispersion from Agent 2)\n# =============================================================================\ncvs = [e.get(\"cv_minutes\", 0.0) for e in A2[\"estimates\"]]\nn_samples = [e.get(\"n_samples\", 0) for e in A2[\"estimates\"]]\nreliability = {\n    \"n_rollouts\": A2[\"n_rollouts\"],\n    \"mean_cv_minutes\": round(float(np.mean(cvs)), 3),\n    \"median_cv_minutes\": round(float(np.median(cvs)), 3),\n    \"pct_nodes_low_cv\": round(float(np.mean([c <= 0.25 for c in cvs]) * 100), 1),\n    \"pct_nodes_full_sampled\":\n        round(float(np.mean([s == A2[\"n_rollouts\"] for s in n_samples]) * 100), 1),\n}\nprint(\"[AXIS 2 \u00b7 Reliability]\")\nprint(f\"   rollouts              : {reliability['n_rollouts']}\")\nprint(f\"   mean CV (minutes)     : {reliability['mean_cv_minutes']}\")\nprint(f\"   nodes with CV<=0.25   : {reliability['pct_nodes_low_cv']}%\")\n\n\n# %%\n# =============================================================================\n# Cell 6. AXIS 3 \u2014 EFFICIENCY (pipeline cost vs. labor value produced)\n#\n# The true cumulative LLM cost is read from the shared cost ledger that every\n# notebook appends to via COST.flush(). If the ledger is absent (e.g. a cached\n# re-run with no fresh calls), we fall back to a small observed constant and say\n# so. The axis reports a ONE-TIME analysis cost against a RECURRING annual\n# saving, so we present both the raw ratio and a plain-language framing rather\n# than leaning on a single dramatic number.\n# =============================================================================\nannual_saving = A3[\"baseline\"][\"annual_saving_usd\"]\n\nLEDGER = ARTIFACTS / \"cost_ledger.json\"\nif LEDGER.exists():\n    ledger = json.loads(LEDGER.read_text(encoding=\"utf-8\"))\n    pipeline_cost = round(sum(v.get(\"total_usd\", 0.0) for v in ledger.values()), 6)\n    total_calls = int(sum(v.get(\"n_calls\", 0) for v in ledger.values()))\n    cost_source = \"measured (cost_ledger.json across all notebooks)\"\nelse:\n    ledger = {}\n    pipeline_cost = 0.02   # conservative observed floor if ledger not built yet\n    total_calls = None\n    cost_source = \"fallback constant (run notebooks with COST.flush to measure)\"\n\n# Guard against a zero/near-zero denominator producing an absurd ratio.\nratio = round(annual_saving / pipeline_cost, 0) if pipeline_cost > 1e-6 else None\n\nefficiency = {\n    \"annual_saving_usd\": round(annual_saving, 0),\n    \"pipeline_llm_cost_usd\": pipeline_cost,\n    \"pipeline_llm_calls\": total_calls,\n    \"cost_source\": cost_source,\n    \"per_stage_cost\": {k: v.get(\"total_usd\") for k, v in ledger.items()},\n    \"saving_to_cost_ratio\": ratio,\n    \"framing\": \"A one-time analysis costing cents produces an estimated \"\n               \"recurring annual labor saving in the five figures; the axis \"\n               \"measures order-of-magnitude leverage, not a precise multiple.\",\n}\nprint(\"[AXIS 3 \u00b7 Efficiency]\")\nprint(f\"   annual saving (USD)   : ${efficiency['annual_saving_usd']:,.0f}\")\nprint(f\"   pipeline LLM cost     : ${efficiency['pipeline_llm_cost_usd']:.4f} \"\n      f\"({cost_source})\")\nif efficiency[\"per_stage_cost\"]:\n    print(f\"   per-stage cost        : {efficiency['per_stage_cost']}\")\nprint(f\"   saving : cost ratio   : \"\n      f\"{efficiency['saving_to_cost_ratio']:,.0f} : 1\"\n      if ratio else \"   saving : cost ratio   : n/a\")\n\n\n# %%\n# =============================================================================\n# Cell 7. AXIS 4 \u2014 TRANSPARENCY (rationale coverage + source-tag provenance)\n# =============================================================================\nrationale_present = [bool((u.get(\"auto_rationale\") or \"\").strip()) for u in units]\nsrc_dist = {}\nfor e in A2[\"estimates\"]:\n    s = e.get(\"source\", \"prior\")\n    src_dist[s] = src_dist.get(s, 0) + 1\ntransparency = {\n    \"grade_rationale_coverage_pct\":\n        round(float(np.mean(rationale_present) * 100), 1),\n    \"time_source_distribution\": src_dist,\n    \"pct_estimates_grounded\":   # implied or stated (i.e. not bare prior)\n        round(float(sum(v for k, v in src_dist.items() if k in (\"stated\", \"implied\"))\n                    / max(sum(src_dist.values()), 1) * 100), 1),\n}\nprint(\"[AXIS 4 \u00b7 Transparency]\")\nprint(f\"   grade rationale coverage : {transparency['grade_rationale_coverage_pct']}%\")\nprint(f\"   time source distribution : {transparency['time_source_distribution']}\")\nprint(f\"   grounded (not bare prior): {transparency['pct_estimates_grounded']}%\")\n\n\n# %%\n# =============================================================================\n# Cell 8. AXIS 5 \u2014 ROBUSTNESS (clarifying questions + unknown handling)\n# =============================================================================\nclar = A2.get(\"clarifying_questions\", [])\nmissing_actor = sum(1 for u in units if (u.get(\"actor\") == MISSING))\nmissing_system = sum(1 for u in units if (u.get(\"system\") == MISSING))\nrobustness = {\n    \"clarifying_questions\": len(clar),\n    \"clarifying_rate_pct\": round(len(clar) / max(len(units), 1) * 100, 1),\n    \"missing_actor_flagged\": missing_actor,\n    \"missing_system_flagged\": missing_system,\n    \"note\": \"The interview stated no explicit times; the pipeline surfaced \"\n            \"low-confidence high-impact nodes as questions instead of \"\n            \"silently trusting weak estimates.\",\n}\nprint(\"[AXIS 5 \u00b7 Robustness]\")\nprint(f\"   clarifying questions   : {robustness['clarifying_questions']} \"\n      f\"({robustness['clarifying_rate_pct']}% of nodes)\")\nprint(f\"   [MISSING] actor flagged: {robustness['missing_actor_flagged']}\")\nprint(f\"   [MISSING] system flagged: {robustness['missing_system_flagged']}\")\n\n\n# %%\n# =============================================================================\n# Cell 9. Assemble the five-axis report (design artifact; no pass/fail verdict)\n#\n# Per the paper, adequacy thresholds are intentionally NOT asserted here; the\n# framework reports evidence per axis and leaves threshold standardization to\n# future confirmatory studies.\n# =============================================================================\nreport = {\n    \"artifact\": \"five_axis_validation_report\",\n    \"stance\": \"protocol demonstrated on one illustrative case; thresholds are \"\n              \"left as future standardization work.\",\n    \"axes\": {\n        \"1_accuracy\": accuracy,\n        \"2_reliability\": reliability,\n        \"3_efficiency\": efficiency,\n        \"4_transparency\": transparency,\n        \"5_robustness\": robustness,\n    },\n}\nOUT = INFER / \"five_axis_report.json\"\nOUT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\nprint(f\"[INFO] Five-axis report -> {rel(OUT)}\")\n\n# Flat summary table for the paper.\n# Guard every formatted value against None so a missing ledger / metric never\n# crashes the summary table.\n_ratio = efficiency.get(\"saving_to_cost_ratio\")\n_ratio_str = f\"{_ratio:,.0f}:1\" if isinstance(_ratio, (int, float)) else \"n/a\"\n\nsummary_rows = [\n    {\"axis\": \"1 Accuracy\",\n     \"key_metric\": \"grade kappa / time MAPE\",\n     \"value\": (f\"kappa={accuracy.get('grade_cohen_kappa')}, \"\n               f\"MAPE={accuracy.get('time_MAPE_pct')}%\") if GROUND_TRUTH else \"n/a\"},\n    {\"axis\": \"2 Reliability\", \"key_metric\": \"mean CV (minutes)\",\n     \"value\": reliability[\"mean_cv_minutes\"]},\n    {\"axis\": \"3 Efficiency\", \"key_metric\": \"saving : cost ratio\",\n     \"value\": _ratio_str},\n    {\"axis\": \"4 Transparency\", \"key_metric\": \"rationale coverage / grounded\",\n     \"value\": f\"{transparency['grade_rationale_coverage_pct']}% / \"\n              f\"{transparency['pct_estimates_grounded']}%\"},\n    {\"axis\": \"5 Robustness\", \"key_metric\": \"clarifying-question rate\",\n     \"value\": f\"{robustness['clarifying_rate_pct']}%\"},\n]\nsdf = pd.DataFrame(summary_rows)\nscsv = TAB / \"five_axis_summary.csv\"\nsdf.to_csv(scsv, index=False, encoding=\"utf-8-sig\")\nprint(f\"[INFO] Five-axis summary table -> {rel(scsv)}\")\nwith pd.option_context(\"display.max_colwidth\", 60, \"display.width\", 160):\n    print(sdf.to_string(index=False))\nprint(f\"[INFO] Axis-1 matcher spend this run: ${COST.total_usd():.5f}\")",
}

_ACC = {"1_accuracy": ["grade_agreement","grade_cohen_kappa","time_MAE_min",
                       "time_MAPE_pct","matched_pairs","unmatched_reference"],
        "2_reliability": ["mean_cv_minutes","median_cv_minutes","pct_nodes_low_cv"],
        "3_efficiency": ["annual_saving_usd","pipeline_llm_cost_usd","saving_to_cost_ratio"],
        "4_transparency": ["grade_rationale_coverage_pct","pct_estimates_grounded"],
        "5_robustness": ["clarifying_rate_pct","missing_system_flagged"]}
def _flatten(rep):
    o={}
    for ax,keys in _ACC.items():
        for k in keys: o[f"{ax}.{k}"]=rep["axes"][ax].get(k)
    return o

_per_tier={t:[] for t in _tiers}
_failures=[]
try:
    for _tier in _tiers:
        for _seed in _seeds:
            print(f"\n=== tier={_tier} seed={_seed} ===", flush=True)
            # set manifest for this run
            _m=_json.loads(_MANIF.read_text(encoding="utf-8"))
            _m["default_tier"]=_tier; _m["seed"]=_seed
            for _st in _m.get("pipeline_cfg",{}): _m["pipeline_cfg"][_st]["tier"]=_tier
            _MANIF.write_text(_json.dumps(_m,ensure_ascii=False,indent=2),encoding="utf-8")
            # per-run globals visible to the exec'd logic
            global _RUN_SEED, _RUN_DISABLE_CACHE
            _RUN_SEED=str(_seed)
            _RUN_DISABLE_CACHE=bool(CONFIG["disable_cache"])
            try:
                for _name in ["01_agent1_task_extraction","015_agent1_5_process_graph",
                              "02_agent2_time_estimation","03_agent3_roi_computation",
                              "04_five_axis_validation"]:
                    print(f"   run {_name}", flush=True)
                    exec(compile(_AGENT_LOGIC[_name], f"<{_name}>", "exec"), globals())
                # save this run
                _dest=_RESULTS/_tier/f"run_{_seed}"; _dest.mkdir(parents=True,exist_ok=True)
                _shutil.copy2(_INFER/"five_axis_report.json", _dest/"five_axis_report.json")
                _rep=_json.loads((_dest/"five_axis_report.json").read_text(encoding="utf-8"))
                _per_tier[_tier].append(_flatten(_rep))
                _a=_rep["axes"]["1_accuracy"]
                print(f"   -> kappa={_a.get('grade_cohen_kappa')} MAPE={_a.get('time_MAPE_pct')}%", flush=True)
            except Exception as _e:
                import traceback as _tb
                _failures.append((_tier,_seed,repr(_e)))
                print(f"   [SKIPPED tier={_tier} seed={_seed}] {type(_e).__name__}: {_e}", flush=True)
                _tb.print_exc()
finally:
    _MANIF.write_text(_ORIG_MANIFEST, encoding="utf-8")
    print("\n[restored original run_manifest.json]")

# aggregate
def _agg(vals):
    nums=[v for v in vals if isinstance(v,(int,float))]
    if not nums: return {"mean":None,"std":None,"n":0}
    return {"mean":round(_stats.mean(nums),4),
            "std":round(_stats.pstdev(nums),4) if len(nums)>1 else 0.0,"n":len(nums)}
_metrics=sorted({k for runs in _per_tier.values() for r in runs for k in r})
_summary={"generated_utc":_dt.now(_tz.utc).isoformat(),"seeds":_seeds,
          "matcher_tier":_MATCHER_TIER,"tiers":{}}
_rows=[]
for _t in _tiers:
    _summary["tiers"][_t]={}
    for _mn in _metrics:
        _a=_agg([r.get(_mn) for r in _per_tier[_t]])
        _summary["tiers"][_t][_mn]=_a
        _rows.append({"tier":_t,"metric":_mn,"mean":_a["mean"],"std":_a["std"],"n":_a["n"]})
(_RESULTS/"tier_comparison.json").write_text(_json.dumps(_summary,ensure_ascii=False,indent=2),encoding="utf-8")
with (_RESULTS/"tier_comparison.csv").open("w",newline="",encoding="utf-8-sig") as _f:
    _w=_csv.DictWriter(_f,fieldnames=["tier","metric","mean","std","n"]); _w.writeheader(); _w.writerows(_rows)
print("\nWrote experiments/results/tier_comparison.json and .csv")
if _failures:
    print(f"\n[WARN] {len(_failures)} run(s) failed:")
    for _t,_s,_e in _failures: print(f"   tier={_t} seed={_s}: {_e[:150]}")
else:
    print("\n[OK] all runs completed.")
print("\nHeadline (mean across runs):")
for _t in _tiers:
    _tt=_summary["tiers"][_t]
    print(f"  {_t:6s} kappa={_tt.get('1_accuracy.grade_cohen_kappa',{}).get('mean')} "
          f"MAPE={_tt.get('1_accuracy.time_MAPE_pct',{}).get('mean')}% "
          f"(kappa std={_tt.get('1_accuracy.grade_cohen_kappa',{}).get('std')})")


[INFO] Manifest loaded. default_tier=weak, grades=['full', 'partial', 'manual']
[INFO] Schema, CostTracker, llm_call() re-established for nb 01.

=== tier=weak seed=42 ===
   run 01_agent1_task_extraction
[INFO] Interview: 5062 chars from data\raw\interview_tcb.txt
[INFO] Agent-1 prompt ready (system 2042 chars, user 5195 chars).
[INFO] Agent 1 returned 41 work units (cached=False, cost=$0.00274).
[INFO] Model reasoning (preview): I analyzed the transcript to identify distinct tasks performed by various actors, ensuring to merge duplicates and split compound tasks into atomic units. Each task was classified based on the information provided regarding who performs it and what systems or tools are used, while also assessing the...
[INFO] Normalized 41 work units.
[INFO] All records passed the validation gate.
 id                                            name                   actor                               system auto_grade                                                          